In [2]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(data.table)

In [80]:
library(argparse)
library(data.table)

# simple method to shuffle knockouts
shuffle_knockouts <- function(d){
    d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
    d$pKO <- ifelse((d$KO == 1), 1,
             ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
             ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0)))
    return(d$pKO)
}

# make header of VCF file
make_vcf_dosage_header <- function(chrom){
    vcf_format <- '##fileformat=VCFv4.2'
    vcf_entry <-  '##FORMAT=<ID=DS,Number=1,Type=Float,Description="">'
    vcf_filter <- '##FILTER=<ID=PASS,Description="All filters passed">"'
    vcf_contig <- paste0('##contig=<ID=',chrom,',length=81195210>')
    vcf_out <- paste(vcf_format, vcf_entry, vcf_filter, vcf_contig, sep = '\n')
    return(vcf_out)
}

# make vcf-like rows for dosage entries
make_vcf_dosage_rows <- function(chrom, positions, marker){
    return(data.table(
      "#CHROM" = chrom,
      POS = positions,
      ID = marker,
      REF = 'X',
      ALT = 'Y',
      QUAL = '.',
      FILTER = '.',
      INFO = '.',
      FORMAT = 'DS'
    ))
}

In [81]:
args <- list(
    input_path = "data/permute/genes/chr1/ukb_eur_wes_200k_pLoF_damaging_missense_chr1_ENSG00000132763.tsv.gz",
    permutations = 10,
    seed = 45
)

In [96]:
#print(args)
stopifnot(file.exists(args$input_path))
stopifnot(!is.na(as.numeric(args$permutations)))
stopifnot(!is.null(args$permutations))

# seed for reproducibility
seed <- as.numeric(args$seed)
set.seed(seed)

# replicate knockout
n <- as.numeric(args$permutations)
d <- fread(args$input_path)
stopifnot(nrow(d) > 0)
reps <- replicate(n, shuffle_knockouts(d))
rownames(reps) <- d$s
reps <- data.table(t(reps)) * 2
rowSums(reps)

[1] 2 2 2 0 2 4 6 2 4 2

In [97]:
sum(reps*2> 1)

[1] 13

In [98]:
kos <- d[d$pTKO > 0]; print(kos)

           gene_id       s unphased_het phased_het hom_alt_n het pTKO
1: ENSG00000132763 2657939            0          2         0   2  0.5
2: ENSG00000132763 2817220            0          2         0   2  0.5
3: ENSG00000132763 3809313            0          2         0   2  0.5
4: ENSG00000132763 4382702            0          2         0   2  0.5


In [99]:
table(replicate(100, sum(rbinom(n=nrow(d), size=1, prob = d$pTKO))))


 0  1  2  3  4 
 5 24 38 27  6 

In [100]:
for (i in 1:10){
d <- fread(args$input_path)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0)))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])
}

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2817220            0          2         0   2  0.5  1   1  2
2: ENSG00000132763 3809313            0          2         0   2  0.5  1   1  2
           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 4382702            0          2         0   2  0.5  1   1  2
Empty data.table (0 rows and 10 cols): gene_id,s,unphased_het,phased_het,hom_alt_n,het...
           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2817220            0          2         0   2  0.5  1   1  2
2: ENSG00000132763 4382702            0          2         0   2  0.5  1   1  2
Empty data.table (0 rows and 10 cols): gene_id,s,unphased_het,phased_het,hom_alt_n,het...
           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2657939            0          2         0   2  0.5  1   1  2
2: ENSG00000132763 2

In [132]:
# test results when two singletons are in the data
set.seed(3)
s_random <- sample(d$s, 1)
selected <- d$s %in% s_random
d$unphased_het[selected] <- 2
d$phased_het[selected] <- 0
d$het[selected] <- d$unphased_het[selected] + d$phased_het[selected]
d$pTKO[selected] <- 0.5
d[selected,]

gene_id,s,unphased_het,phased_het,hom_alt_n,het,pTKO,KO,pKO,DS
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
ENSG00000132763,2497053,2,0,0,2,0.5,1,0.5,1


In [125]:
# scenario: knockout = 0 and DS = 1
set.seed(1)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2497053            2          0         0   2  0.5  0 0.5  1
2: ENSG00000132763 2817220            0          2         0   2  0.5  1 1.0  2
3: ENSG00000132763 3809313            0          2         0   2  0.5  1 1.0  2


In [126]:
# scenario: knockout = 1 and DS should still be 1
set.seed(4)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2497053            2          0         0   2  0.5  1 0.5  1
2: ENSG00000132763 4382702            0          2         0   2  0.5  1 1.0  2


In [128]:
# test homs as well
set.seed(99)
s_random <- sample(d$s, 1)
selected <- d$s %in% s_random
d$hom_alt_n[selected] <- 1
d$pTKO[selected] <- 1
d[selected,]

gene_id,s,unphased_het,phased_het,hom_alt_n,het,pTKO,KO,pKO,DS
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
ENSG00000132763,4661283,0,0,1,0,1,0,0,0


In [129]:
# scenario: knockout = 1 and DS should still be 1
set.seed(4)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2497053            2          0         0   2  0.5  1 0.5  1
2: ENSG00000132763 4382702            0          2         0   2  0.5  1 1.0  2
3: ENSG00000132763 4661283            0          0         1   0  1.0  1 1.0  2


In [133]:
# one phased het and one unphased het
set.seed(66331)
s_random <- sample(d$s, 1)
selected <- d$s %in% s_random
d$unphased_het[selected] <- 1
d$phased_het[selected] <- 1
d$het[selected] <- d$unphased_het[selected] + d$phased_het[selected]
d$pTKO[selected] <- 0.5
d[selected,]

gene_id,s,unphased_het,phased_het,hom_alt_n,het,pTKO,KO,pKO,DS
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
ENSG00000132763,2731264,1,1,0,2,0.5,0,0,0


In [134]:
set.seed(4)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO pKO DS
1: ENSG00000132763 2497053            2          0         0   2  0.5  1 0.5  1
2: ENSG00000132763 2731264            1          1         0   2  0.5  0 0.5  1
3: ENSG00000132763 3809313            0          2         0   2  0.5  1 1.0  2
4: ENSG00000132763 4661283            0          0         1   0  1.0  1 1.0  2


In [143]:
# one phased het and two unphased het
set.seed(66514)
s_random <- sample(d$s, 1)
selected <- d$s %in% s_random
d$unphased_het[selected] <- 2
d$phased_het[selected] <- 1
d$het[selected] <- d$unphased_het[selected] + d$phased_het[selected]
d$pTKO[selected] <- 1 - 2*(1/2) ^ d$het[selected]
d[selected,]

gene_id,s,unphased_het,phased_het,hom_alt_n,het,pTKO,KO,pKO,DS
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
ENSG00000132763,1109463,2,1,0,3,0.75,0,0,0


In [153]:
# what does the DS look like when KO == 0 ?
set.seed(7)
d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
d$DS <- d$pKO * 2
print(d[d$pKO > 0, ])

           gene_id       s unphased_het phased_het hom_alt_n het pTKO KO  pKO
1: ENSG00000132763 1109463            2          1         0   3 0.75  0 0.75
2: ENSG00000132763 2497053            2          0         0   2 0.50  0 0.50
3: ENSG00000132763 2731264            1          1         0   2 0.50  0 0.50
4: ENSG00000132763 3809313            0          2         0   2 0.50  1 1.00
5: ENSG00000132763 4661283            0          0         1   0 1.00  1 1.00
    DS
1: 1.5
2: 1.0
3: 1.0
4: 2.0
5: 2.0
